# Лекція 7: Дизайн взаємодії агентів

**Курс**: Мульти-агентні системи

---

> 💡 **Що ви дізнаєтесь у цій лекції:**
> - Як обирати оптимальний патерн взаємодії агентів (кооперативний, ієрархічний, конкурентний) під конкретну задачу
> - Як проєктувати систему ролей для мультиагентної команди (Planner, Executor, Critic, Evaluator)
> - Як реалізувати основні патерни взаємодії в LangGraph (supervisor, collaboration, hierarchical teams, plan-and-execute)
> - Що таке "coordination tax" — коли мультиагентність шкодить, а не допомагає
> - Основи event-driven підходу до мультиагентних систем

In [ ]:
!pip install -q langchain==1.2.13 langchain-openai==1.1.11 langgraph==1.1.3 langgraph-supervisor==0.0.31 python-dotenv==1.2.2 pydantic==2.12.5 ddgs

In [ ]:
import os
import warnings
from getpass import getpass
from typing import Annotated
from operator import add


warnings.filterwarnings("ignore")

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

print("API key configured!")

# LLM setup — used across all examples
from langchain.chat_models import init_chat_model

# Change the model name if needed (e.g., "anthropic:claude-sonnet-4-6-20250929")
LLM_POWERFUL = "openai:gpt-5.4"       # for planning, evaluation, supervision
LLM_FAST = "openai:gpt-5.4-nano"     # for execution, simple tasks

llm = init_chat_model(LLM_POWERFUL)
llm_fast = init_chat_model(LLM_FAST)

print(f"✅ Models configured: powerful={LLM_POWERFUL}, fast={LLM_FAST}")

---

## 1. Таксономія патернів взаємодії

Лекція 6 дала огляд **фреймворків оркестрації** — як працює кожен інструмент окремо. Тепер ми переходимо до **архітектурних патернів взаємодії між агентами**: як проєктувати мультиагентну систему незалежно від конкретного фреймворку.

Перш ніж заглиблюватися — ось повна карта патернів, які ми розглянемо:

| Патерн | Джерело | Суть (одним реченням) |
|--------|---------|----------------------|
| **Sequential / Prompt Chaining** | Anthropic + Microsoft | Лінійний pipeline: вихід кроку N → вхід кроку N+1 |
| **Parallelization / Concurrent** | Anthropic + Microsoft | Незалежні підзадачі виконуються паралельно |
| **Orchestrator-Workers** | Anthropic + Microsoft | Центральний LLM динамічно розбиває задачу і делегує |
| **Evaluator-Optimizer** | Anthropic | Цикл: генерація → оцінка → покращення |
| **Handoff** | Microsoft (+ LangChain) | Агенти самі передають контроль один одному |
| **Group Chat** | Microsoft | Розмовний формат зі спільним тредом, включає maker-checker |
| **Magentic-One** | Microsoft Research | Динамічний task ledger, план будується в процесі |

### 1.1. Спектр складності — recap

Нагадування з Лекції 6 — коли мультиагентність потрібна, а коли ні:

| Рівень | Коли використовувати |
|--------|---------------------|
| **Direct model call** | Однокрокові задачі (класифікація, переклад) |
| **Single agent + tools** | Різнорідні запити в одному домені |
| **Multi-agent orchestration** | Крос-доменні задачі, різні security boundaries, паралельна спеціалізація |

> 🔑 **Ключова ідея:** Починай з найпростішого рішення і нарощуй складність лише коли це доказово покращує результат. Кожен рівень вносить coordination overhead, latency та cost.

Сьогодні ми переходимо від питання "чи потрібна мультиагентність?" до "який **патерн** взаємодії обрати?"

### 1.2. Класифікація workflow-патернів (Anthropic)

Anthropic виділяє п'ять основних workflow-патернів, які лежать в основі будь-якої мультиагентної системи:

**1. Prompt Chaining (послідовний ланцюжок)**
- Задача розбивається на послідовні кроки, кожен LLM-виклик обробляє вихід попереднього
- Між кроками можна додавати програмні перевірки (gates)
- **Приклад:** Генерація маркетингового тексту → переклад → адаптація під аудиторію
- **LangGraph аналог:** лінійний граф з послідовними нодами

**2. Routing (маршрутизація)**
- Вхід класифікується і направляється до спеціалізованого обробника
- Дозволяє розділити concerns і побудувати оптимізовані промпти для кожного типу
- **Приклад:** Customer support — загальні питання / рефанди / технічна підтримка
- **LangGraph аналог:** conditional edges після класифікатора

**3. Parallelization (паралелізація)**
- Два варіанти: **Sectioning** (розбиття на незалежні підзадачі) та **Voting** (кілька запусків для різноманітності)
- **Приклад Sectioning:** Один агент обробляє запит, інший одночасно перевіряє на inappropriate content
- **LangGraph аналог:** `Send` API, паралельні гілки графа

**4. Orchestrator-Workers (оркестратор-виконавці)**
- Центральний LLM динамічно розбиває задачу, делегує worker-ам, синтезує результати
- Ключова відмінність від паралелізації: підзадачі **НЕ** визначені заздалегідь
- **Приклад:** Coding agent, що змінює кілька файлів залежно від задачі

**5. Evaluator-Optimizer (оцінювач-оптимізатор)**
- Один LLM генерує відповідь, інший оцінює і дає фідбек у циклі
- **Приклад:** Літературний переклад з ітеративним покращенням
- **LangGraph аналог:** maker-checker loop, цикл з conditional edge

![Anthropic Workflow Patterns 1-3](attachment:anthropic_patterns_1_3.png)

![Anthropic Workflow Patterns 4-5](attachment:anthropic_patterns_4_5.png)

**Що ми бачимо на діаграмах:**

- **Prompt Chaining** — найпростіший патерн: лінійний pipeline з gates між кроками для програмних перевірок
- **Routing** — fan-out по типу вхідного запиту; кожен handler оптимізований під свій тип
- **Parallelization** — незалежні задачі виконуються паралельно і агрегуються
- **Orchestrator-Workers** — центральний LLM динамічно розподіляє роботу; двосторонні стрілки показують, що workers повертають результати orchestrator-у
- **Evaluator-Optimizer** — цикл генерації та оцінки, який завершується коли якість досягає порогу

> 📊 Ці п'ять патернів покривають більшість потреб. Вони комбінуються: наприклад, Orchestrator може делегувати worker-ам, кожен з яких внутрішньо використовує Prompt Chaining.

### 1.3. Розширена таксономія оркестрації (Microsoft Azure)

Microsoft виділяє п'ять оркестраційних патернів. Вони частково перетинаються з патернами Anthropic, але додають нові перспективи:

| Microsoft патерн | Anthropic аналог | Що нового |
|------------------|------------------|-----------|
| **Sequential** | ≈ Prompt Chaining | Та сама ідея — лінійний pipeline |
| **Concurrent** | ≈ Parallelization | Та сама ідея — паралельні агенти |
| **Handoff** | Частково Orchestrator-Workers | Агенти **самі** ініціюють передачу контролю |
| **Group Chat** | — (нова концепція) | Розмовний формат зі спільним тредом |
| **Magentic-One** | — (нова концепція) | Динамічний task ledger |

> 🔑 **Ключова ідея:** **Handoff vs Orchestrator-Workers** — в обох є координація, але в Orchestrator-Workers центральний LLM приймає всі рішення. В Handoff агенти **самі** ініціюють передачу контролю. Концепція походить з OpenAI Swarm і тепер нативно підтримується в LangGraph через `Command`-based state updates.

**Зв'язок з LangChain:** У поточній таксономії LangChain (2026) виділяються чотири основних патерни: **Subagents** (рекомендований за замовчуванням), **Handoffs**, **Skills** та **Router**.

**Деталі нових патернів Microsoft:**

| Патерн | Координація | Маршрутизація | Найкраще для | Ризики |
|--------|-------------|---------------|-------------|--------|
| **Sequential** | Лінійний pipeline | Детерміновано | Покрокове покращення з чіткими залежностями | Помилки каскадуються |
| **Concurrent** | Паралельно | Детерміновано або динамічно | Незалежний аналіз з різних перспектив | Потрібна resolution стратегія |
| **Group Chat** | Розмовний формат | Chat manager | Консенсус, brainstorming, maker-checker | Складно з >3 агентів |
| **Handoff** | Один активний агент | Агенти самі вирішують | Спеціаліст з'ясовується в процесі | Нескінченні петлі |
| **Magentic-One** | Plan-build-execute | Менеджер динамічно | Відкриті проблеми без відомого шляху | Повільне сходження |

**Magentic-One** (Fourney et al., Microsoft Research, 2024) — мультиагентна система, де Orchestrator Agent будує і динамічно оновлює "task ledger", розподіляючи задачі між спеціалізованими агентами (WebSurfer, FileSurfer, Coder, ComputerTerminal). Ключова відмінність: план **не відомий заздалегідь**.

**Group Chat / Maker-Checker loop:** Один агент (maker) створює артефакт, інший (checker) оцінює за чіткими критеріями. Потрібен iteration cap і fallback. Рекомендація Microsoft: не більше 3 агентів у group chat.

> 📊 Між двома таксономіями — ~7 унікальних патернів. Sequential, Concurrent і Orchestrator-Workers — спільні. Handoff, Group Chat, Magentic-One та Evaluator-Optimizer — додаткові.

### 1.4. Виміри взаємодії агентів

Патерни описують **як** організувати потік роботи. Але є ортогональне питання: **з якою метою** агенти взаємодіють?

| Вимір взаємодії | Суть | Типові патерни | Ключовий виклик |
|-----------------|------|----------------|-----------------|
| **Кооперативна** | Агенти працюють на спільну мету | Collaboration, Supervisor, Plan-and-Execute | Синхронізація контексту |
| **Ієрархічна** | Чітка ієрархія: supervisor → workers | Supervisor, Hierarchical Teams | Bottleneck на supervisor-рівні |
| **Конкурентна (debate)** | Агенти займають протилежні позиції | Group Chat, debate-графи | Потрібен resolution mechanism |

**Приклади:**
- **Кооперативна:** Research team — один шукає інформацію, інший аналізує, третій пише звіт
- **Ієрархічна:** Менеджер проєкту розподіляє задачі між командами
- **Конкурентна:** Investment committee — Bull analyst vs Bear analyst → Chairman вирішує

> 🔑 **Ключова ідея:** Спершу визнач **вимір** взаємодії (кооперація? ієрархія? дебат?), потім обери **патерн** з таксономії.

![Як обрати патерн взаємодії агентів](attachment:decision_tree.png)

---

## 2. Ролі в мультиагентній системі та "AI Teams"

Тепер, коли ми знаємо **патерни**, потрібно зрозуміти **з кого** складається мультиагентна команда. Як розподілити відповідальність між агентами, щоб кожен робив одну річ добре?

### 2.1. Приклад: AI Team — Planner–Coder–Reviewer

Класична "AI-команда" для software development:

```
User Request → Planner → Coder → Reviewer → (loop if needed) → Final Output
```

- **Planner:** Отримує задачу, розбиває на конкретні кроки реалізації
- **Coder:** Пише код за планом, використовує інструменти (file write, shell exec)
- **Reviewer:** Перевіряє код на помилки, стиль, відповідність плану. Якщо є проблеми — повертає Coder-у

Важливі design decisions:
- Чи бачить Reviewer весь контекст (план + код), чи тільки код?
- Скільки ітерацій допускається перед ескалацією до людини?
- Як передається контекст між агентами — повністю або compact summary?

In [ ]:
# Planner-Coder-Reviewer team as a LangGraph graph

from langchain.agents import create_agent
from langchain_core.tools import tool
from langgraph.graph import StateGraph, MessagesState, START, END
from pydantic import BaseModel
from typing import Literal

@tool
def write_file(path: str, content: str) -> str:
    """Write content to a file."""
    return f"File {path} written successfully."

@tool
def run_tests(path: str) -> str:
    """Run tests for a file."""
    return "All tests passed."

# Define agents with different model tiers
planner = create_agent(
    model=LLM_POWERFUL, 
    tools=[],
    system_prompt=(
        "You are a software architect. Given a task, break it into 2-3 concrete implementation steps. "
        "Be concise — just a numbered list."
    ),
    name="planner",
)

coder = create_agent(
    model=LLM_FAST, 
    tools=[write_file, run_tests],
    system_prompt="You are a Python developer. Implement the plan step by step. Be concise.",
    name="coder",
)

reviewer = create_agent(
    model=LLM_POWERFUL, 
    tools=[],
    system_prompt=(
        "You are a code reviewer. Check code for bugs and plan compliance. "
        "If issues found, say REVISION_NEEDED and list problems. "
        "If code is good, say APPROVED."
    ),
    name="reviewer",
)

# Reviewer routing: return string, not Command (conditional edges require strings)
def review_router(state: MessagesState) -> Literal["coder", "__end__"]:
    """Route based on reviewer verdict."""
    last_msg = state["messages"][-1].content
    if "APPROVED" in last_msg.upper():
        return END
    return "coder"

# Build the graph: planner → coder → reviewer → (loop or end)
graph = StateGraph(MessagesState)
graph.add_node("planner", planner)
graph.add_node("coder", coder)
graph.add_node("reviewer", reviewer)
graph.add_edge(START, "planner")
graph.add_edge("planner", "coder")
graph.add_edge("coder", "reviewer")
graph.add_conditional_edges("reviewer", review_router)

dev_team_app = graph.compile()
print("✅ Planner-Coder-Reviewer graph compiled")

# Run the team
result = dev_team_app.invoke(
    {"messages": [{"role": "user", "content": "Create a Python function to check if a number is prime"}]},
    {"recursion_limit": 25},
)

# Print each agent's contribution
for msg in result["messages"][1:]:  # skip the user message
    name = getattr(msg, "name", msg.type)
    print(f"\n{'='*60}")
    print(f"🤖 {name}:")
    print(msg.content)

In [ ]:
from IPython.display import Image
Image(dev_team_app.get_graph().draw_mermaid_png())

### 2.2. Приклад: Investment Committee (конкурентна взаємодія)

Мультиагентна система для аналізу інвестицій — приклад **structured debate pattern**:

- **Bull Analyst:** Шукає позитивні сигнали, growth потенціал, сильні фінансові показники
- **Bear Analyst:** Ідентифікує ризики, негативні тренди, regulatory headwinds
- **Chairman:** Зважує аргументи обох сторін, приймає фінальне рішення

Порядок:
1. Bull agent починає з інвестиційного кейсу
2. Bear agent контраргументує
3. Chairman синтезує і виносить рішення

In [ ]:
# Investment Committee: structured debate with sequential agents

from langgraph.graph import StateGraph, MessagesState, START, END

from ddgs import DDGS

@tool
def web_search(query: str) -> str:
    """Search the web for information using DuckDuckGo."""
    results = DDGS().text(query, max_results=3)
    return "\n".join(f"- {r['title']}: {r['body']}" for r in results) if results else "No results found."

bull = create_agent(
    model=LLM_FAST, tools=[web_search],
    system_prompt=(
        "You are a BULL analyst. Present the strongest case FOR investing. "
        "Focus on: growth potential, strong financials, competitive advantages. Be concise (3-5 sentences)."
    ),
    name="bull_analyst",
)

bear = create_agent(
    model=LLM_FAST, tools=[web_search],
    system_prompt=(
        "You are a BEAR analyst. Present the strongest case AGAINST investing. "
        "Focus on: risks, overvaluation, competitive threats. Be concise (3-5 sentences)."
    ),
    name="bear_analyst",
)

chairman = create_agent(
    model=LLM_POWERFUL, tools=[],
    system_prompt=(
        "You are the chairman of an investment committee. "
        "Weigh both sides and make a final BUY / HOLD / SELL recommendation. Be concise."
    ),
    name="chairman",
)

# Sequential debate: bull → bear → chairman
graph = StateGraph(MessagesState)
graph.add_node("bull_analyst", bull)
graph.add_node("bear_analyst", bear)
graph.add_node("chairman", chairman)
graph.add_edge(START, "bull_analyst")
graph.add_edge("bull_analyst", "bear_analyst")
graph.add_edge("bear_analyst", "chairman")
graph.add_edge("chairman", END)

debate_app = graph.compile()
print("✅ Investment Committee debate graph compiled")

# Run the debate
result = debate_app.invoke({"messages": [{"role": "user", "content": "Analyze NVIDIA as an investment."}]})

# Print each agent's contribution
for msg in result["messages"][1:]:
    name = getattr(msg, "name", msg.type)
    print(f"\n{'='*60}")
    print(f"🤖 {name}:")
    print(msg.content)

In [ ]:
from IPython.display import Image
Image(debate_app.get_graph().draw_mermaid_png())

### 2.3. Практичні принципи розподілу ролей

1. **Один агент = одна відповідальність.** Не створюйте "universal" агент, який робить все
2. **Не створюйте агентів без потреби.** Кожен додатковий агент — це coordination overhead
3. **Чітко визначайте input/output контракт** кожного агента (що отримує, що повертає, в якому форматі)
4. **Використовуйте structured output** для комунікації між агентами (Pydantic models, JSON schemas)
5. **Давайте агентам різні моделі** під задачу: потужну для planning/evaluation, дешевшу для execution

In [ ]:
# Structured output for inter-agent communication

from pydantic import BaseModel

class PlanOutput(BaseModel):
    """Contract between Planner and Executor."""
    steps: list[str]
    estimated_complexity: str  # "simple" | "medium" | "complex"
    required_tools: list[str]

class ReviewOutput(BaseModel):
    """Contract between Reviewer and the system."""
    verdict: str  # "APPROVED" | "REVISION_NEEDED"
    issues: list[str]
    score: float  # 0.0 — 1.0

# Planner returns a structured plan
plan = llm.with_structured_output(PlanOutput).invoke(
    [{"role": "user", "content": "Create a REST API endpoint for user registration"}]
)
print("Plan:")
print(f"  Steps: {plan.steps}")
print(f"  Complexity: {plan.estimated_complexity}")
print(f"  Required tools: {plan.required_tools}")

# Reviewer returns a structured review
review = llm.with_structured_output(ReviewOutput).invoke(
    [{"role": "user", "content": "Review this code: def add(a, b): return a + b"}]
)
print(f"\nReview:")
print(f"  Verdict: {review.verdict}")
print(f"  Score: {review.score}")
print(f"  Issues: {review.issues}")

---

## 3. Реалізація патернів у LangGraph

Тепер переходимо до hands-on імплементації. В Лекції 6 ми вже вивчили `StateGraph`, nodes, edges, `MessagesState`, conditional edges та checkpointing. Тут ми фокусуємось **лише на нових концепціях**, потрібних для мультиагентних систем.

### 3.1. Нові інструменти LangGraph для мультиагентності

Ключові **нові** концепції поверх Лекції 6:

**`Command`** — об'єднує оновлення стану та переходу до іншого вузла. Дозволяє ноді одночасно оновити state і вказати, куди рухатися далі.

**`create_agent`** — високорівнева фабрика для створення ReAct-агента. Агент отримує модель, tools та system prompt, повертає скомпільований LangGraph граф.

**Subgraphs** — вкладені графи як ноди для ієрархічних команд. Дозволяють композицію графів.

**`Send` API** — для map-reduce та паралельної обробки динамічної кількості задач.

In [ ]:
# Command API + base multi-agent graph

from langgraph.types import Command
from langgraph.graph import StateGraph, MessagesState, START, END
from langchain_core.messages import AIMessage
from typing import Literal

# Router model for supervisor structured output
class Router(BaseModel):
    """Supervisor routing decision."""
    next: Literal["researcher", "writer", "FINISH"]
    reasoning: str

# Create agents
researcher = create_agent(
    model=LLM_FAST, 
    tools=[web_search],
    system_prompt="You are a web researcher. Find information and be concise.",
    name="researcher",
)

writer = create_agent(
    model=LLM_FAST, 
    tools=[],
    system_prompt="You are a technical writer. Write clear, structured text. Be concise.",
    name="writer",
)

# Supervisor uses Command API: route + update state in one step
def supervisor_node(state: MessagesState) -> Command[Literal["researcher", "writer", "__end__"]]:
    response = llm.with_structured_output(Router).invoke(
        [{"role": "system", "content": "Route to researcher or writer. Say FINISH when done."}]
        + state["messages"]
    )
    goto = END if response.next == "FINISH" else response.next
    return Command(goto=goto, update={"messages": [AIMessage(content=f"[Supervisor] {response.reasoning}")]})

graph = StateGraph(MessagesState)
graph.add_node("supervisor", supervisor_node)
graph.add_node("researcher", researcher)
graph.add_node("writer", writer)
graph.add_edge(START, "supervisor")
graph.add_edge("researcher", "supervisor")
graph.add_edge("writer", "supervisor")

base_app = graph.compile()
print("✅ Base multi-agent graph with Command API compiled")

# Run it
result = base_app.invoke(
    {"messages": [{"role": "user", "content": "Research what LangGraph is and write a one-paragraph summary"}]},
    {"recursion_limit": 15},
)

# Print each agent's contribution
for msg in result["messages"][1:]:
    name = getattr(msg, "name", msg.type)
    print(f"\n{'='*60}")
    print(f"🤖 {name}:")
    print(msg.content[:500] if len(msg.content) > 500 else msg.content)

In [ ]:
from IPython.display import Image
Image(base_app.get_graph().draw_mermaid_png())

### 3.2. Патерн 1: Multi-Agent Collaboration (спільний scratchpad)

**Концепція:** Два або більше агентів працюють по черзі над спільним станом (shared scratchpad). Кожен бачить весь контекст попередньої роботи. Routing — через функцію, яка визначає наступного агента.

**Приклад:** Research Agent + Chart Agent
- Research Agent: шукає дані за допомогою web search
- Chart Agent: створює візуалізації на основі знайдених даних
- Router: визначає, хто працює наступний

> ⚠️ Router у цьому прикладі використовує keyword matching — це **лише для демонстрації**. У production router — це окремий LLM-call з structured output (як у Патерні 2 — Supervisor).

In [ ]:
# Pattern 1: Multi-Agent Collaboration with shared scratchpad

import matplotlib.pyplot as plt
import re

@tool
def create_chart(data: str) -> str:
    """Create a bar chart from data. Input: 'label1:value1, label2:value2, ...' or free-text with numbers."""
    # Try to parse "label:value" pairs
    pairs = re.findall(r'([\w\s/.-]+?)\s*[:\-–]\s*([\d.]+)', data)
    if not pairs:
        return f"Could not parse chart data from: {data}"

    labels = [p[0].strip() for p in pairs]
    values = [float(p[1]) for p in pairs]

    fig, ax = plt.subplots(figsize=(8, 4))
    colors = ['#4C78A8', '#F58518', '#54A24B', '#E45756', '#72B7B2', '#B279A2', '#EECA3B']
    ax.barh(labels, values, color=colors[:len(labels)])
    ax.set_xlabel("Value")
    ax.set_title("Generated Chart")
    for i, v in enumerate(values):
        ax.text(v + max(values)*0.02, i, f"{v:g}", va='center', fontsize=10)
    plt.tight_layout()
    plt.show()
    return f"Bar chart created with {len(labels)} items: {', '.join(labels)}"

research_agent = create_agent(
    model=LLM_FAST, 
    tools=[web_search],
    system_prompt="You are a research agent. Search for data and summarize findings. Be concise.",
    name="researcher",
)

chart_agent = create_agent(
    model=LLM_FAST, 
    tools=[create_chart],
    system_prompt="You are a chart agent. Create visualizations from research data. Be concise.",
    name="chart_generator",
)

# Simple sequential router: research first, then chart, then done
# ⚠️ DEMO ONLY — in production use LLM-based routing (see Pattern 2)
def router(state: MessagesState):
    agent_names = [getattr(m, "name", None) for m in state["messages"]]
    if "chart_generator" in agent_names:
        return END  # both agents have worked — done
    if "researcher" in agent_names:
        return "chart_agent"  # research done — now visualize
    return "research_agent"  # start with research

graph = StateGraph(MessagesState)
graph.add_node("research_agent", research_agent)
graph.add_node("chart_agent", chart_agent)
graph.add_conditional_edges(START, router)
graph.add_conditional_edges("research_agent", router)
graph.add_conditional_edges("chart_agent", router)

collab_app = graph.compile()
print("✅ Collaboration pattern compiled")

# Run it
result = collab_app.invoke(
    {"messages": [{"role": "user", "content": "Search for Python web framework popularity and create a chart with the data"}]}
)

# Print each agent's contribution
for msg in result["messages"][1:]:
    name = getattr(msg, "name", msg.type)
    print(f"\n{'='*60}")
    print(f"🤖 {name}:")
    print(msg.content[:500] if len(msg.content) > 500 else msg.content)

In [ ]:
from IPython.display import Image
Image(collab_app.get_graph().draw_mermaid_png())

### 3.3. Патерн 2: Supervisor (централізована координація)

**Концепція:** Supervisor-агент вирішує, який worker повинен працювати наступним. Workers — це окремі агенти зі своїми інструментами. Supervisor використовує structured output для вибору.

Розглянемо **три підходи** до реалізації:

In [ ]:
# Pattern 2A: Supervisor via langgraph-supervisor library

from langgraph_supervisor import create_supervisor

@tool
def vector_search(query: str) -> str:
    """Search internal documents using vector similarity."""
    return f"Internal docs for '{query}': [Company Q3 revenue: $2.1B, up 23% YoY, gross margin 68%]"

web_researcher = create_agent(
    model=LLM_FAST, 
    tools=[web_search],
    system_prompt="You are a web researcher. Search and summarize external information. Be concise.",
    name="web_researcher",
)

rag_agent = create_agent(
    model=LLM_FAST, 
    tools=[vector_search],
    system_prompt="You are a RAG agent. Find relevant internal documents. Be concise.",
    name="rag_agent",
)

workflow = create_supervisor(
    [web_researcher, rag_agent],
    model=init_chat_model(LLM_FAST),
    prompt="You are a team supervisor. Route questions to the right specialist.",
)

supervisor_a_app = workflow.compile()
print("✅ Approach A: langgraph-supervisor compiled")

# Run it
result = supervisor_a_app.invoke({"messages": [{"role": "user", "content": "What are our Q3 results?"}]})

# Print each agent's contribution
for msg in result["messages"][1:]:
    name = getattr(msg, "name", msg.type)
    print(f"\n{'='*60}")
    print(f"🤖 {name}:")
    print(msg.content[:500] if len(msg.content) > 500 else msg.content)

In [ ]:
from IPython.display import Image
Image(supervisor_a_app.get_graph().draw_mermaid_png())

In [ ]:
# Pattern 2B: Supervisor via Command API (full control)

class SupervisorRouter(BaseModel):
    """Supervisor routing decision."""
    next: Literal["web_researcher", "rag_agent", "FINISH"]
    reasoning: str

def supervisor_b_node(state: MessagesState) -> Command[Literal["web_researcher", "rag_agent", "__end__"]]:
    system_prompt = (
        "You are a supervisor. Decide which agent should handle the request.\n"
        "- web_researcher: for external, public information\n"
        "- rag_agent: for internal company documents\n"
        "- FINISH: when the task is complete"
    )
    response = llm.with_structured_output(SupervisorRouter).invoke(
        [{"role": "system", "content": system_prompt}] + state["messages"]
    )
    goto = END if response.next == "FINISH" else response.next
    return Command(goto=goto)

graph_b = StateGraph(MessagesState)
graph_b.add_node("supervisor", supervisor_b_node)
graph_b.add_node("web_researcher", web_researcher)
graph_b.add_node("rag_agent", rag_agent)
graph_b.add_edge(START, "supervisor")
graph_b.add_edge("web_researcher", "supervisor")
graph_b.add_edge("rag_agent", "supervisor")

supervisor_b_app = graph_b.compile()
print("✅ Approach B: Command API supervisor compiled")

# Run it
result = supervisor_b_app.invoke(
    {"messages": [{"role": "user", "content": "Compare our Q3 results with industry benchmarks"}]},
    {"recursion_limit": 15},
)

# Print each agent's contribution
for msg in result["messages"][1:]:
    name = getattr(msg, "name", msg.type)
    print(f"\n{'='*60}")
    print(f"🤖 {name}:")
    print(msg.content[:500] if len(msg.content) > 500 else msg.content)

In [ ]:
from IPython.display import Image
Image(supervisor_b_app.get_graph().draw_mermaid_png())

In [ ]:
# Pattern 2C: Subagents-as-Tools (recommended by LangChain for new projects)

@tool
def research_web(query: str) -> str:
    """Search the public web for external information on a topic."""
    result = web_researcher.invoke({"messages": [{"role": "user", "content": query}]})
    return result["messages"][-1].content

@tool
def search_internal_docs(query: str) -> str:
    """Search internal company documents using RAG."""
    result = rag_agent.invoke({"messages": [{"role": "user", "content": query}]})
    return result["messages"][-1].content

supervisor_c = create_agent(
    model=LLM_POWERFUL,
    tools=[research_web, search_internal_docs],
    system_prompt=(
        "You are a team supervisor. Use research_web for external information "
        "and search_internal_docs for internal company documents. "
        "Synthesize results into a clear answer. Be concise."
    ),
    name="supervisor",
)

print("✅ Approach C: Subagents-as-Tools compiled (recommended for new projects)")

# Run it
result = supervisor_c.invoke(
    {"messages": [{"role": "user", "content": "What are our Q3 results vs industry average?"}]},
    {"recursion_limit": 15},
)

# Print each agent's contribution
for msg in result["messages"][1:]:
    name = getattr(msg, "name", msg.type)
    print(f"\n{'='*60}")
    print(f"🤖 {name}:")
    print(msg.content[:500] if len(msg.content) > 500 else msg.content)

**Порівняння підходів до Supervisor:**

| | Підхід A (бібліотека) | Підхід B (Command API) | Підхід C (subagents-as-tools) |
|---|---|---|---|
| **Контроль** | Мінімальний | Максимальний | Середній |
| **Context isolation** | Слабка | Потрібна вручну | Автоматична |
| **Рекомендація LangChain** | Backward compat | Складні кастомні графи | ✅ Рекомендований для нових проєктів |
| **Коли обирати** | Швидкий прототип | Нестандартна routing-логіка | Більшість production use cases |

**Переваги supervisor-патерну** (незалежно від підходу): чіткий контроль потоку, легко додавати нових агентів.

**Недоліки:** Supervisor — single point of failure, bottleneck.

### 3.4. Патерн 3: Hierarchical Agent Teams

**Концепція:** Агенти в нодах — це самі по собі LangGraph-графи (subgraphs). Це дозволяє створювати ієрархічні команди:

```
Top Supervisor
├── Research Team (sub-supervisor)
│   ├── Web Search Agent
│   └── Paper Analysis Agent
└── Writing Team (sub-supervisor)
    ├── Draft Writer Agent
    └── Editor Agent
```

**Переваги:** Масштабованість, розділення concerns.
**Недоліки:** Складність дебагу, можлива втрата контексту між рівнями.

In [ ]:
# Pattern 3a: Hierarchical Teams — Research Team subgraph

from langchain_core.messages import HumanMessage

class ResearchRouter(BaseModel):
    """Research team routing. Return exactly one JSON object."""
    next: Literal["search", "analysis", "FINISH"]

class WritingRouter(BaseModel):
    """Writing team routing. Return exactly one JSON object."""
    next: Literal["draft_writer", "editor", "FINISH"]

class TopRouter(BaseModel):
    """Top-level routing. Return exactly one JSON object."""
    next: Literal["research_team", "writing_team", "FINISH"]

search_agent = create_agent(
    model=LLM_FAST, tools=[web_search],
    system_prompt="Search the web for information. Be concise.", name="search",
)
analysis_agent = create_agent(
    model=LLM_FAST, tools=[],
    system_prompt="Analyze and summarize research findings in 2-3 sentences.", name="analysis",
)

def research_supervisor(state: MessagesState) -> Command[Literal["search", "analysis", "__end__"]]:
    agent_names = [getattr(m, "name", None) for m in state["messages"]]
    # Simple heuristic: search first, then analysis, then finish
    if "analysis" in agent_names:
        return Command(goto=END)
    if "search" in agent_names:
        return Command(goto="analysis")
    return Command(goto="search")

research_graph = StateGraph(MessagesState)
research_graph.add_node("supervisor", research_supervisor)
research_graph.add_node("search", search_agent)
research_graph.add_node("analysis", analysis_agent)
research_graph.add_edge(START, "supervisor")
research_graph.add_edge("search", "supervisor")
research_graph.add_edge("analysis", "supervisor")
research_team = research_graph.compile()
print("✅ Research Team subgraph compiled (search → analysis)")

# Run the research team subgraph independently
result = research_team.invoke(
    {"messages": [{"role": "user", "content": "Find information about LangGraph framework"}]},
    {"recursion_limit": 15},
)

# Print each agent's contribution
for msg in result["messages"][1:]:
    name = getattr(msg, "name", msg.type)
    print(f"\n{'='*60}")
    print(f"🤖 {name}:")
    print(msg.content[:500] if len(msg.content) > 500 else msg.content)

In [ ]:
from IPython.display import Image
Image(research_team.get_graph().draw_mermaid_png())

In [ ]:
# Pattern 3b: Writing Team + Top-level supervisor

draft_writer = create_agent(
    model=LLM_FAST, tools=[],
    system_prompt="Write a first draft based on the research. Keep it to 1-2 paragraphs.", name="draft_writer",
)
editor_agent = create_agent(
    model=LLM_FAST, tools=[],
    system_prompt="Edit and improve the draft for clarity and style. Keep it concise.", name="editor",
)

def writing_supervisor(state: MessagesState) -> Command[Literal["draft_writer", "editor", "__end__"]]:
    agent_names = [getattr(m, "name", None) for m in state["messages"]]
    # Simple heuristic: draft first, then edit, then finish
    if "editor" in agent_names:
        return Command(goto=END)
    if "draft_writer" in agent_names:
        return Command(goto="editor")
    return Command(goto="draft_writer")

writing_graph = StateGraph(MessagesState)
writing_graph.add_node("supervisor", writing_supervisor)
writing_graph.add_node("draft_writer", draft_writer)
writing_graph.add_node("editor", editor_agent)
writing_graph.add_edge(START, "supervisor")
writing_graph.add_edge("draft_writer", "supervisor")
writing_graph.add_edge("editor", "supervisor")
writing_team = writing_graph.compile()

# Top-level supervisor uses LLM to decide which team works next
def top_supervisor_node(state: MessagesState) -> Command[Literal["research_team", "writing_team", "__end__"]]:
    response = llm.with_structured_output(TopRouter).invoke(
        [{"role": "system", "content": (
            "You are a top-level supervisor managing two teams: research_team and writing_team. "
            "Given the conversation, decide which team should work next. "
            "Typical flow: research_team first, then writing_team, then FINISH. "
            "Return a single JSON with the 'next' field."
        )}]
        + state["messages"]
    )
    goto = END if response.next == "FINISH" else response.next
    return Command(goto=goto)

top_graph = StateGraph(MessagesState)
top_graph.add_node("research_team", research_team)
top_graph.add_node("writing_team", writing_team)
top_graph.add_node("top_supervisor", top_supervisor_node)
top_graph.add_edge(START, "top_supervisor")
top_graph.add_edge("research_team", "top_supervisor")
top_graph.add_edge("writing_team", "top_supervisor")

hierarchical_app = top_graph.compile()
print("✅ Hierarchical Agent Teams graph compiled")
print("   Top supervisor → Research Team (search → analysis) + Writing Team (draft → edit)")

# Run it
result = hierarchical_app.invoke(
    {"messages": [{"role": "user", "content": "Research multi-agent systems and write a brief overview"}]},
    {"recursion_limit": 30},
)

# Print each agent's contribution
for msg in result["messages"][1:]:
    name = getattr(msg, "name", msg.type)
    print(f"\n{'='*60}")
    print(f"🤖 {name}:")
    print(msg.content[:500] if len(msg.content) > 500 else msg.content)

In [ ]:
from IPython.display import Image
Image(hierarchical_app.get_graph().draw_mermaid_png())

### 3.5. Патерн 4: Plan-and-Execute

**Концепція:** Спочатку Planner створює повний план виконання. Потім Executor послідовно виконує кожен крок. Після кожного кроку Replanner може скоригувати залишок плану.

```
Input → Planner → [Step1 → Executor → Replanner → Step2 → Executor → ...] → Final Response
```

**Переваги:** Явний план, можливість human review перед виконанням, replanning.
**Недоліки:** Overhead на планування, план може бути неадекватним для open-ended задач.

> 🔑 **Ключова ідея:** `results` має reducer `Annotated[list[str], add]` — це забезпечує **акумулювання** результатів кроків, а не перезапис. Без reducer кожен виклик executor затирав би попередні результати.

In [ ]:
# Pattern 4a: Plan-and-Execute — state and models

from typing_extensions import TypedDict
from langgraph.graph.message import add_messages

executor_agent = create_agent(
    model=LLM_FAST, tools=[web_search],
    system_prompt="Execute the given step using available tools.", name="executor",
)

class Plan(BaseModel):
    steps: list[str]

class ReplanDecision(BaseModel):
    """Replanner decision: whether to adjust remaining steps."""
    adjusted_steps: list[str] | None = None

class PlanState(TypedDict):
    messages: Annotated[list, add_messages]
    plan: list[str]
    current_step: int
    results: Annotated[list[str], add]  # accumulates instead of overwriting

print("✅ Plan-and-Execute state and models defined")

In [ ]:
# Pattern 4b: Plan-and-Execute — node functions and graph

def planner_node(state: PlanState) -> dict:
    plan = llm.with_structured_output(Plan).invoke(
        [{"role": "system", "content": "Break the task into 3-5 concrete steps."}]
        + state["messages"]
    )
    return {"plan": plan.steps, "current_step": 0}

def executor_node(state: PlanState) -> dict:
    if state["current_step"] >= len(state["plan"]):
        return {"results": ["No remaining steps."]}
    step = state["plan"][state["current_step"]]
    result = executor_agent.invoke({"messages": [HumanMessage(content=step)]})
    return {
        "results": [result["messages"][-1].content],
        "current_step": state["current_step"] + 1,
    }

def replanner_node(state: PlanState) -> Command[Literal["executor", "__end__"]]:
    if state["current_step"] >= len(state["plan"]):
        return Command(goto=END)
    replan_prompt = (
        f"Original plan: {state['plan']}\n"
        f"Completed steps: {state['current_step']}\n"
        f"Results so far: {state['results']}\n"
        f"Remaining steps: {state['plan'][state['current_step']:]}\n\n"
        "Should we continue with the current plan, or adjust remaining steps?"
    )
    replan_decision = llm.with_structured_output(ReplanDecision).invoke(
        [{"role": "user", "content": replan_prompt}]
    )
    if replan_decision.adjusted_steps:
        new_plan = state["plan"][:state["current_step"]] + replan_decision.adjusted_steps
        return Command(goto="executor", update={"plan": new_plan})
    return Command(goto="executor")

plan_graph = StateGraph(PlanState)
plan_graph.add_node("planner", planner_node)
plan_graph.add_node("executor", executor_node)
plan_graph.add_node("replanner", replanner_node)
plan_graph.add_edge(START, "planner")
plan_graph.add_edge("planner", "executor")
plan_graph.add_edge("executor", "replanner")

plan_app = plan_graph.compile()
print("✅ Plan-and-Execute graph compiled")
print("   Flow: Planner → Executor → Replanner → (loop or end)")

# Run it
result = plan_app.invoke(
    {"messages": [{"role": "user", "content": "Research the top 3 Python web frameworks and compare them"}]},
    {"recursion_limit": 25},
)
print("\n📝 Plan executed. Results:")
for i, r in enumerate(result.get("results", [])):
    print(f"  Step {i+1}: {r[:200]}...")

In [ ]:
from IPython.display import Image
Image(plan_app.get_graph().draw_mermaid_png())

---

## 4. Coordination Tax та практичні рекомендації

Тепер, коли ми вміємо будувати мультиагентні системи, потрібно зрозуміти: **коли це корисно, а коли — ні?** В Лекції 6 ми давали інтуїтивне правило. Тепер підкріпимо його **емпіричними даними**.

### 4.1. The Multi-Agent Coordination Tax

**Дослідження 1: "Towards a Science of Scaling Agent Systems" (Kim et al., Google Research, MIT, UW et al., 2025)**

Масштабне дослідження: 180 конфігурацій агентів, 4 бенчмарки, 3 сім'ї LLM.

**Формула продуктивності:**
```
Net Performance = (Individual Capability + Collaboration Benefits)
                - (Coordination Chaos + Communication Overhead + Tool Complexity)
```

> ⚠️ Наступні числові результати — орієнтири з конкретного дослідження, а не універсальні закони.

**Ключові знахідки:**
1. **Topology matters:** Централізована координація покращувала продуктивність на **80.8%** для паралелізовних задач. Але для послідовних reasoning задач **всі** мультиагентні конфігурації **погіршили** результат на 39–70%
2. **Error Amplification:** Незалежні MAS підсилювали помилки до **17.2x**. Централізовані — до **4.4x**
3. **Task decomposability:** Найбільший виграш — на задачах з незалежними паралельними підзадачами

**Дослідження 2: MAST — Multi-Agent System Failure Taxonomy (Cemri et al., UC Berkeley / NeurIPS 2025)**
- 1600+ трейсів у 7 open-source MAS-фреймворках
- 14 типів збоїв у 3 категоріях: system design issues, inter-agent misalignment, task verification failures

![Compound Reliability Decay & Token Cost](attachment:reliability_and_cost.png)

**Що ми бачимо на графіках:**

- **Compound Reliability Decay** (ліворуч): навіть при 95% надійності кожного кроку, система з 20 кроків має лише ~36% шанс на успіх. Це означає: **мінімізуйте кількість кроків** і додавайте перевірки на кожному етапі
- **Token Cost Multiplication** (праворуч): кожен додатковий агент збільшує витрати на ~60% через shared context і coordination overhead. Документний аналіз з 1 агентом (10K tokens) потребує ~35K з 4 агентами

**Compound reliability decay (95% per step):**
-  5 steps → 77.4% system success
- 10 steps → 59.9% system success
- 15 steps → 46.3% system success
- 20 steps → 35.8% system success

> 🔑 **Ключова ідея:** Мультиагентність — це trade-off. Виграш у спеціалізації має перевищувати coordination tax.

### 4.2. Коли використовувати мультиагентність — Decision Framework

**✅ Використовуй мультиагентну систему, коли:**
- Задача природно розбивається на **незалежні паралельні** підзадачі
- Потрібні **різні security boundaries** для різних частин системи
- Один агент не справляється через prompt complexity або tool overload
- Потрібні **різні моделі** для різних етапів (cost optimization)
- Задача потребує **structured debate** (multiple perspectives)

**❌ НЕ використовуй мультиагентну систему, коли:**
- Один агент з правильним промптом і tools справляється
- Задача **послідовна і тісно зв'язана** — кожен крок залежить від повного контексту
- Latency критична і overhead координації неприйнятний
- Single-agent baseline вже показує високу accuracy

**Чек-лист перед вибором патерну:**

1. Чи можна вирішити одним model call? → **Зупинись тут**
2. Чи потрібні tools/reasoning? → **Один агент + tools**
3. Чи є незалежні підзадачі? → **Concurrent / Parallelization**
4. Чи є чіткий pipeline? → **Sequential / Prompt Chaining**
5. Чи потрібна динамічна маршрутизація? → **Supervisor / Handoff**
6. Чи потрібен debate/consensus? → **Group Chat / Adversarial**
7. Чи невідомий план заздалегідь? → **Plan-and-Execute / Magentic-One**

### 4.3. Практичні рекомендації

1. **Context management:** Вирішуйте, що передавати наступному агенту — повний контекст чи compact summary
2. **Error handling:** Timeout + retry + graceful degradation. Валідуйте output перед передачею
3. **Security:** Кожен агент — principle of least privilege. Security trimming per user
4. **Cost optimization:** Різні моделі для різних агентів. Monitor token consumption per agent
5. **Observability:** Інструментуйте ВСІ операції і handoff-и. Використовуйте LangSmith/Langfuse для трейсингу

---

## 🎯 Підсумок

### Що ми вивчили:

1. **Таксономія патернів** — ~7 унікальних патернів із двох таксономій (Anthropic + Microsoft): Sequential, Parallelization, Routing, Orchestrator-Workers, Evaluator-Optimizer, Handoff, Group Chat, Magentic-One
2. **Ролі агентів** — 6 категорій (Control, Planning, Execution, Assurance, Context, Human Oversight) для структурованого проєктування
3. **Реалізація в LangGraph** — 4 патерни hands-on: Collaboration, Supervisor (3 підходи), Hierarchical Teams, Plan-and-Execute
4. **Coordination Tax** — мультиагентність не завжди краще: compound reliability decay, error amplification, token cost multiplication
5. **Event-Driven MAS** — preview: Orchestrator-Worker та Blackboard patterns через Kafka

---

## 📝 Вправи

### Вправа 1 (🟢 Easy): Evaluator-Optimizer Loop

Реалізуйте простий **Evaluator-Optimizer** цикл у LangGraph:
1. **Maker** генерує короткий текст (summary) за заданою темою
2. **Checker** оцінює якість тексту і повертає score (0.0–1.0)
3. Якщо score < 0.8 і ітерація < 3 — повертаємось до Maker з feedback
4. Інакше — завершуємо

**Підказка:** Використовуйте `with_structured_output` для Checker і `Command` для routing.

In [ ]:
# Exercise 1: Evaluator-Optimizer loop

class EvalResult(BaseModel):
    """Checker evaluation output."""
    score: float
    feedback: str

# TODO: Define the state
# class EvalState(TypedDict):
#     ...

# TODO: Define maker node
# def maker(state):
#     ...

# TODO: Define checker node
# def checker(state) -> Command:
#     ...

# TODO: Build the graph
# graph = StateGraph(EvalState)
# ...

# Validation:
# result = app.invoke({"messages": [{"role": "user", "content": "Write a summary about multi-agent systems"}]})
# assert len(result.get("results", [])) > 0, "Should have at least one result"
# print("✅ Exercise 1 passed!")

### Вправа 2 (🟡 Medium): Subagents-as-Tools Supervisor

Створіть supervisor-агента з **трьома** sub-agent tools (Підхід C з §3.3):
1. `code_writer` — пише Python код за описом
2. `code_reviewer` — рев'юїть код і знаходить проблеми
3. `test_writer` — пише unit-тести для наданого коду

Supervisor повинен координувати роботу: спочатку написати код, потім зробити review і написати тести.

**Підказка:** Кожен sub-agent загортається як `@tool` функція.

In [ ]:
# Exercise 2: Subagents-as-Tools with 3 sub-agents

# TODO: Create the code_writer agent
# code_writer = create_agent(...)

# TODO: Create the code_reviewer agent
# code_reviewer = create_agent(...)

# TODO: Create the test_writer agent
# test_writer = create_agent(...)

# TODO: Wrap each agent as a @tool
# @tool
# def write_code(description: str) -> str:
#     ...

# TODO: Create the supervisor
# supervisor = create_agent(
#     model=LLM_POWERFUL,
#     tools=[write_code, review_code, write_tests],
#     ...
# )

# Validation:
# result = supervisor.invoke({"messages": [{"role": "user", "content": "Create a function to validate email addresses"}]})
# print(result["messages"][-1].content)
# print("✅ Exercise 2 passed!")

### Вправа 3 (🔴 Hard): Динамічний Plan-and-Execute з Send API

Розширте Патерн 4 (Plan-and-Execute) з використанням **Send API** для паралельного виконання незалежних кроків:

1. **Planner** створює план і позначає, які кроки можуть виконуватись паралельно
2. Кроки, що не залежать один від одного, виконуються **одночасно** через `Send`
3. Послідовні кроки виконуються по черзі
4. **Replanner** переглядає результати після кожної "хвилі" паралельних кроків

**Підказка:** Використовуйте `Send` API для fan-out і `Annotated[list, add]` reducer для збору результатів.

**Очікуваний результат:** Час виконання менший, ніж при послідовному виконанні всіх кроків.

In [ ]:
# Exercise 3: Dynamic Plan-and-Execute with Send API

from langgraph.types import Send

# TODO: Define a Plan model with parallel step groups
# class ParallelPlan(BaseModel):
#     waves: list[list[str]]  # each wave = list of parallel steps

# TODO: Define state
# class ParallelPlanState(TypedDict):
#     ...

# TODO: Implement planner that creates waves of parallel steps
# def planner(state):
#     ...

# TODO: Implement fan-out using Send API
# def fan_out_wave(state):
#     current_wave = state["waves"][state["current_wave"]]
#     return [Send("executor", {"task": step}) for step in current_wave]

# TODO: Implement executor for a single step
# def executor(state):
#     ...

# TODO: Build the graph
# graph = StateGraph(ParallelPlanState)
# ...

# Validation:
# result = app.invoke({"messages": [{"role": "user", "content": "..."}]})
# print("✅ Exercise 3 passed!")

---

## 📚 Джерела та додаткові матеріали

1. **Anthropic** — "Building Effective Agents" (2024): [anthropic.com/engineering/building-effective-agents](https://www.anthropic.com/engineering/building-effective-agents)
2. **Microsoft Azure** — "AI Agent Design Patterns" (2026): [learn.microsoft.com](https://learn.microsoft.com/en-us/azure/architecture/ai-ml/guide/ai-agent-design-patterns)
3. **LangGraph** — Graph API docs: [docs.langchain.com](https://docs.langchain.com/oss/python/langgraph/graph-api)
4. **LangChain** — Agents docs (`create_agent`): [docs.langchain.com](https://docs.langchain.com/oss/python/langchain/agents)
5. **LangChain** — Multi-agent patterns overview: [docs.langchain.com](https://docs.langchain.com/oss/python/langchain/multi-agent)
6. **Kim et al.** — "Towards a Science of Scaling Agent Systems" (2025): [arxiv.org/abs/2512.08296](https://arxiv.org/abs/2512.08296)
7. **Cemri et al.** — "Why Do Multi-Agent LLM Systems Fail?" / MAST (NeurIPS 2025): [arxiv.org/abs/2503.13657](https://arxiv.org/abs/2503.13657)
8. **Fourney et al.** — "Magentic-One: A Generalist Multi-Agent System" (2024): [microsoft.com/research](https://www.microsoft.com/en-us/research/articles/magentic-one-a-generalist-multi-agent-system-for-solving-complex-tasks/)
9. **Confluent** — "Four Design Patterns for Event-Driven Multi-Agent Systems": [confluent.io/blog](https://www.confluent.io/blog/event-driven-multi-agent-systems/)
10. **LangGraph Supervisor library**: [github.com/langchain-ai/langgraph-supervisor-py](https://github.com/langchain-ai/langgraph-supervisor-py)